# Forward Kinematics & Inverse Kinematics with Kinova robot

It demonstrates: 
1. Importing required libraries and helpers (Pinocchio, KinovaPy utilities).
2. Helper functions for forward and inverse kinematics.
3. Loading the robot model from the URDF.
4. (Optional) Connecting to a Kinova device to read joint angles and compute FK/IK.

Notes:
- Running the device-connection cells requires the Kinova Kortex API and network access to the robot.
- Example FK/IK calculations can be run locally using Pinocchio with a test `q` if you do not have a device connected.

In [11]:
# Imports
import numpy as np
from numpy.linalg import norm, solve
import pinocchio as pin

In [12]:
from kortex_api.Exceptions.KServerException import KServerException

def get_joint_angles(base):
    # Current arm's joint angles (in home position)
    try:
        print("Getting Angles for every joint...")
        input_joint_angles = base.GetMeasuredJointAngles()
    except KServerException as ex:
        print("Unable to get joint angles")
        print("Error_code:{} , Sub_error_code:{} ".format(ex.get_error_code(), ex.get_error_sub_code()))
        print("Caught expected error: {}".format(ex))
        return False

    print("Joint ID : Joint Angle (deg)")
    for joint_angle in input_joint_angles.joint_angles:
        print(joint_angle.joint_identifier, " : ", joint_angle.value)
    print()

    # Use Pinocchio
    q = np.deg2rad([joint_angle.value for joint_angle in input_joint_angles.joint_angles])

    return q


def compute_forward_kinematics(model, data, q, FRAME_ID):
    # Compute the Forward Kinematics
    print("\nComputing Forward Kinematics...")
    pin.forwardKinematics(model, data, q)
    pin.updateFramePlacements(model, data)
    T_ee = data.oMf[FRAME_ID].copy()
    p_ee = np.append(T_ee.translation, pin.rpy.matrixToRpy(T_ee.rotation))
    return T_ee, p_ee


def compute_inverse_kinematics(model, data, oMdes, FRAME_ID, q_guess=None, verbose=False):
    # Compute the Inverse Kinematics
    print("\nComputing Inverse Kinematics...")
    if q_guess is not None:
        q = q_guess
    else:
        q = pin.neutral(model)
    eps = 1e-3
    IT_MAX = 1000
    DT = 1e-2
    damp = 1e-12

    i = 0
    while True:
        pin.forwardKinematics(model, data, q)
        pin.updateFramePlacements(model, data)
        fMd = data.oMf[FRAME_ID].actInv(oMdes)
        err = pin.log(fMd).vector  # in joint frame
        if norm(err) < eps:
            success = True
            break
        if i >= IT_MAX:
            success = False
            break
        J = pin.computeFrameJacobian(model, data, q, FRAME_ID)  # in joint frame
        J = -np.dot(pin.Jlog6(fMd.inverse()), J)
        v = -J.T.dot(solve(J.dot(J.T) + damp * np.eye(6), err))
        q = pin.integrate(model, q, v * DT)
        if verbose and not i % 10:
            print(f"Iteration {i}: difference = {err.T}")
        i += 1

    if success:
        print("Convergence achieved!")
    else:
        print("\nWarning: the iterative algorithm has not reached convergence to the desired precision")

    return q


## Load robot model (Pinocchio)

Builds the Pinocchio model from the URDF included in the `KinovaPy` package. This cell does not require a robot connection and can be executed locally.

In [13]:
# Load robot model
import pinocchio as pin
from KinovaPy import MESHES_PATH, URDF_PATH
import os

urdf_name = 'GEN3-7DOF-VISION_ARM_URDF_V12.urdf'
urdf_path = os.path.join(URDF_PATH, urdf_name)
meshes_dir = MESHES_PATH
robot = pin.RobotWrapper.BuildFromURDF(urdf_path, meshes_dir, root_joint=None)
rmodel = robot.model
rdata = robot.data
FRAME_ID = rmodel.getFrameId('end_effector_link')
print("Loaded model. FRAME_ID=", FRAME_ID)


Loaded model. FRAME_ID= 17


## Example: compute FK/IK locally using a test configuration

If you do not have a device connected, you can use `pin.neutral(rmodel)` as a joint configuration and compute forward and inverse kinematics locally.

In [14]:
# Example local FK/IK run
q_test = np.array([0.0, 0.5, 0.0, 0.5, 0.0, 0.5, 0.0])  # Example joint angles (rad)
T_ee, p_ee = compute_forward_kinematics(rmodel, rdata, q_test, FRAME_ID)
print("End-effector pose (x,y,z,roll,pitch,yaw):\n", p_ee)
# Use inverse kinematics to recover q from T_ee
q_ik = compute_inverse_kinematics(rmodel, rdata, T_ee, FRAME_ID, q_guess=q_test*0.9, verbose=False)
print("Recovered joint angles (rad):", q_ik)



Computing Forward Kinematics...
End-effector pose (x,y,z,roll,pitch,yaw):
 [ 6.33283443e-01 -2.48557265e-02  8.35756136e-01  1.52109254e-04
  1.50000000e+00  1.55392222e-04]

Computing Inverse Kinematics...
Convergence achieved!
Recovered joint angles (rad): [-4.87830384e-07  4.99753614e-01  1.50934698e-06  4.99583053e-01
 -1.63407533e-06  4.99771998e-01  6.52937961e-07]


## Optional: connect to Kinova device and compute FK/IK from measured joints

The following cell mirrors the original script's device connection. Only run this cell if you have the Kinova Kortex API installed and network access to the robot. The connection uses `KinovaPy.utilities` to parse connection arguments.

In [15]:
# Optional device connection (RUN ONLY IF YOU HAVE DEVICE ACCESS)
from kortex_api.autogen.client_stubs.BaseClientRpc import BaseClient
from KinovaPy import utilities

args = utilities.parseConnectionArguments()
with utilities.DeviceConnection.createTcpConnection(args) as router:
    base = BaseClient(router)
    q = get_joint_angles(base)
    if q is not False:
        print("Joint angles (rad):\n", q)
        T_ee, p_ee = compute_forward_kinematics(rmodel, rdata, q, FRAME_ID)
        print("End-effector pose (x,y,z,roll,pitch,yaw):\n", p_ee)
        q_ik = compute_inverse_kinematics(rmodel, rdata, T_ee, FRAME_ID, q_guess=q*0.9)
        print("Joint angles from IK (rad):\n", q_ik)
    else:
        print("Skipping FK/IK computation due to error in getting joint angles.")

Logging as admin on device 192.168.1.10
Getting Angles for every joint...
Joint ID : Joint Angle (deg)
0  :  0.2650648355484009
1  :  15.006990432739258
2  :  180.0009002685547
3  :  229.99131774902344
4  :  0.0002821807865984738
5  :  54.99537658691406
6  :  90.24336242675781

Joint angles (rad):
 [4.62625411e-03 2.61921394e-01 3.14160837e+00 4.01410575e+00
 4.92498381e-06 9.59850395e-01 1.57504380e+00]

Computing Forward Kinematics...
End-effector pose (x,y,z,roll,pitch,yaw):
 [ 4.56645824e-01 -7.71701507e-04  4.33602288e-01  1.57115060e+00
  4.25044531e-03  1.56615866e+00]

Computing Inverse Kinematics...
Convergence achieved!
Joint angles from IK (rad):
 [ 0.1125255   0.26757975  3.01150147  4.017028   -0.02099967  0.96034095
  1.62113801]
